In [12]:
import spacy
import os
import inflect

model_path = os.getcwd() + '\\model'

model = spacy.load(model_path)

In [13]:
p = inflect.engine()

query = 'Which foods contain luteolin?'

entities = model(query)

sorted_entities = [p.singular_noun(word.text) if not p.singular_noun(word.text) == False else word.text for word in entities.ents]

print(sorted_entities)

['luteolin']


In [14]:
from neo4j import GraphDatabase
import json

with open('go away\\passwords.json', 'r') as f:
    passwords = json.load(f)

neo_password = passwords['neo4j']

del passwords

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", neo_password)

In [ ]:
driver = GraphDatabase.driver(URI, auth=AUTH, max_connection_lifetime=3600)

node_types = ['FOOD', 'COMPOUND', 'CONCENTRATION']
relationship_types = ['CONTAINS', 'FOUNDATCONCENTRATION']
patterns = [
    ('FOOD', 'CONTAINS', 'COMPOUND'),
    ('COMPOUND', 'FOUNDATCONCENTRATION', 'CONCENTRATION')
]

numbers_map = {0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}

def preprocess_label(label):
    if label[0].isnumeric():
        label = label.replace(label[0], numbers_map[int(label[0])])
    replace_chars = ["-", "(", ")", " ", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，", "<", ">"]
    for char in replace_chars:
        label = label.replace(char, '')
    return label

def preprocess_query(query):
    replace_chars = ["-", "(", ")", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，", "<", ">"]
    for char in replace_chars:
        query = query.replace(char, '')
    return query

def preprocess_name(name):
    replace_chars = ["'"]
    for char in replace_chars:
        name = name.replace(char, '')
    return name

entity_context = set()

query_processed = preprocess_query(query)

def process_relation(target):
    target_dict = dict(target)
    target_context_string = ""
    if 'name' in target_dict.keys():
        target_context_string += f"{target_dict['name']}, "
        del target_dict['name']
    for prop, value in target_dict.items():
        if prop == 'customId':
            continue
        target_context_string += f"{prop}: {value}, "
    return target_context_string

for entity in sorted_entities:
    entity_label = preprocess_label(entity)
    entity_name = preprocess_name(entity)
    command = f"MATCH (a)-[r]->(b)-[s]->(c) WHERE LOWER(a.customId) LIKE '{entity_label.lower()}' RETURN a.name AS source_name, type(r) AS reltype1, b AS target, type(s) AS reltype2, c AS target2, properties(r) AS rels1, properties(s) AS rels2"
    records, summary, keys = driver.execute_query(command)
    # Format the query results and append to the result string
    if not records:
        command = f"MATCH (a)-[r]->(b)-[s]->(c) WHERE any(k IN keys(a) WHERE k CONTAINS {entity.lower()} AND a[k] = 'Yes') RETURN a.name AS source_name,type(r) AS reltype1,b AS target, type(s) AS reltype2, c AS target2, properties(r) AS rels1, properties(s) AS rels2"
        records, summary, keys = driver.execute_query(command)
    for record in records:
        target_context_1 = process_relation(record['target'])
        target_context_2 = process_relation(record['target2'])

        entity_context.add(f"\n{record['source_name']} - {record['reltype1']} {record['rels1']} -> {target_context_1[:-2]}")
        entity_context.add(f"\n{record['target'].get('name')} - {record['reltype2']} {record['rels2']} -> {target_context_2[:-2]}")

joined_context = "".join(entity_context)

tokenised_context = model(joined_context)

print(len([token for token in tokenised_context]))

print(joined_context)

Unable to retrieve routing information
Transaction failed and will be retried in 0.9982344275476743s (Unable to retrieve routing information)
Unable to retrieve routing information
Transaction failed and will be retried in 1.6686443589190778s (Unable to retrieve routing information)


KeyboardInterrupt: 

In [ ]:
%%script False

import requests
import os
from huggingface_hub import InferenceClient

model_name = 'BLACKBUN/llama-2-7b-pubmed-qa-211k-gguf_q8_0'
model_name2 = 'clinicalnlplab/finetuned-PMCLLaMA-PubmedSumm' #the one that has the inference provider
model_name3 = 'mustangs/gemma-medical_qa-Finetune'

inference_client = InferenceClient(provider='auto', api_key=os.environ['HF_TOKEN'])

#response = requests.post(f"https://api-inference.huggingface.co/models/{model_name2}", headers={"Authorization":f"Bearer {HF_TOKEN}"}, json={"inputs":final_query}).json()[0]["generated_text"]

response = inference_client.text_generation(query, model=model_name2, max_new_tokens=4095-len(query), repetition_penalty=-1)

print(response)

Couldn't find program: 'False'
